# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# Imports
import os
from dotenv import load_dotenv
import sys
import asyncio
import subprocess
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI
import anthropic

NOTEBOOK_DIR = os.path.dirname(os.path.abspath(globals().get("__vsc_ipynb_file__", os.getcwd())))
SCRAPER_SCRIPT = os.path.join(NOTEBOOK_DIR, "scraper.py")

In [31]:
# constants

MODEL_CLAUDE = 'claude-haiku-4-5'
MODEL_LLAMA = 'llama3.2'

In [32]:
load_dotenv(override=True)

# Claude

claude = anthropic.Anthropic()

#  Llama

llama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


In [ ]:
# here is the question; type over this to ask something new

user_prompt = """
Tell me what your model name is.
"""

system_prompt = "You are answering a simple question in 5 words or less."

messages_llama = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
]

messages_claude = [{"role": "user", "content": user_prompt}]

In [ ]:
# Test Claude

response = claude.messages.create(
    model=MODEL_CLAUDE,
    max_tokens=200,
    system=system_prompt,
    messages=messages_claude)

print(response.content[0].text)

In [ ]:
# Test Llama

response = llama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages_llama)

print(response.choices[0].message.content)

### Portfolio Website Reviewer

In [14]:
# Scraper

headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def _fetch_rendered_html_sync(url):
    """Fetch a page's fully-rendered, unprocessed HTML by running scraper.py in raw mode."""
    result = subprocess.run(
        [sys.executable, SCRAPER_SCRIPT, url, "--mode", "raw"],
        capture_output=True, text=True, encoding="utf-8", check=True,
    )
    return result.stdout

async def _fetch_website_content(url):
    return await asyncio.to_thread(_fetch_rendered_html_sync, url)

class Website:

    def __init__(self, url, html):
        """
        Create this Website object from already-fetched HTML using the BeautifulSoup library.
        Use the async `Website.create(url)` factory below instead of calling this directly.
        """
        self.url = url
        soup = BeautifulSoup(html, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True) if soup.body else ""

    @classmethod
    async def create(cls, url):
        """
        Async factory: fetch the page with requests, then fall back to a headless-browser
        render (via _fetch_rendered_html) if the page looks JS-rendered (empty title/body).
        """
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        title = soup.title.string if soup.title else None
        body_text = soup.body.get_text(separator="\n", strip=True) if soup.body else ""

        html = response.content
        if not title or len(body_text) < 50:
            html = await _fetch_website_content(url)

        return cls(url, html)

In [74]:
# Prompt Functions - Reviewer

def website_analysis_user_prompt_generator(website):
    user_prompt = f"You are looking at a website titled {website.title}."
    user_prompt += f"The website content is as follows."
    user_prompt += website.text
    return user_prompt

# System Prompt

system_prompt_reviewer = "You are a recruiting support assistant that analyzes the content of a personal portfolio website and provides insight into\
                what's working well, what isn't, and what career positions and roles the candidate is well positioned for. Follow with advice on how the user can further\
                improve their portfolio to support their goals. Start by trying to understand what roles the user is gunning for through their website, conduct research into\
                what stands out for those roles, and then start providing insights and advice. Make sure to consider the fact that unless this is a website a candidate is trying to\
                compare to, they are likely a recent graduate that is\
                confused about the current job market and provide strategic guidance to finding employment. This guidance should be specific, directly based on their strengths, weaknesses, abilities,\
                and the state of the job market for their targeted roles. Do not hallucinate, base every insight strictly on the content of the website."


# Messages

async def claude_reviewer_user_prompt(url):
    website = await Website.create(url)
    prompt_text = website_analysis_user_prompt_generator(website)
    user_prompt = [{"role": "user", "content": prompt_text}]

    return user_prompt

async def llama_user_prompt(url):
    
    website = await Website.create(url)
    user_prompt = website_analysis_user_prompt_generator(website)

    return user_prompt

async def llama_messages_info(system_prompt, user_prompt):

    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}]
    
    return messages

async def llama_reviewer_messages_builder(url):

    user_prompt = await llama_user_prompt(url)
    messages = await llama_messages_info(system_prompt_reviewer, user_prompt)

    return messages

In [54]:
async def portfolio_reviewer_claude(url):
    user_prompt = await claude_reviewer_user_prompt(url)
    response = claude.messages.create(
         model=MODEL_CLAUDE,
        max_tokens=10000,
        system=system_prompt_reviewer,
        messages=user_prompt)

    summary = response.content[0].text

    display(Markdown(summary))

    return summary

async def portfolio_reviewer_llama(url):

    messages = await llama_reviewer_messages_builder(url)
    response = llama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages)

    summary = response.choices[0].message.content

    display(Markdown(summary))

    return summary

    

In [ ]:
await portfolio_reviewer_llama("https://www.reneesingh.com")

In [ ]:
await portfolio_reviewer_claude("https://www.reneesingh.com")

### Comparing Tool

In [44]:
# I already have a function for reviewing one portfolio
# Now I need to add a comparer

# First, return the review of your portfolio, then return the review of a second portfolio. Then, run an API call to compare and contrast.


In [75]:
# Prompting

system_prompt_comparer = "You are a recruiting support assistant that has recieved two reports from another assistant\
                         about the contents of two personal portfolio websites, each providing insight into what's working\
                         well, what isn't, and what career positions and roles the candidate is well positioned for. This\
                         report also includes an analysis of waht roles the user is gunning for, market research on those\
                         roles, and specific insights and advice targeted at new graduates in the roles they are looking for.\
                         Your job is to compare these two websites. Website 1 is the candidate's website, and Website 2 is another user's website the\
                         website they are trying to compare theirs with. Provide the candidate insights on what's working well for the other\
                         user, what isn't, how relevant this portfolio is as inspiration for theirs based on their backgrounds\
                         and goals, what the user can adapt into their own portfolio to improve it, and what to avoid. Make sure to be\
                         specific. Do not hallucinate, base every insight strictly on the content of the websites."

def website_comparing_user_prompt_generator(summary1, summary2):

    user_prompt = "This is the analysis report for website 1:"
    user_prompt += summary1
    user_prompt += "This is the analysis report for website 2:"
    user_prompt += summary2
    
    return user_prompt

async def llama_comparer_messages_builder(summary1, summary2):

    user_prompt = website_comparing_user_prompt_generator(summary1, summary2)
    messages = await llama_messages_info(system_prompt_comparer, user_prompt)

    return messages

def claude_comparer_user_prompt_generator(summary1, summary2):

    user_prompt = website_comparing_user_prompt_generator(summary1, summary2)
    messages = [{"role": "user", "content": user_prompt}]

    return messages


In [76]:
async def compare_resumes_llama(url1, url2):

    summary1 = await portfolio_reviewer_llama(url1)
    summary2 = await portfolio_reviewer_llama(url2)
    messages = await llama_comparer_messages_builder(summary1, summary2)
    
    response = llama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages)

    summary = response.choices[0].message.content

    display(Markdown(summary))

async def compare_resumes_claude(url1, url2):

    summary1 = await portfolio_reviewer_claude(url1)
    summary2 = await portfolio_reviewer_claude(url2)
    user_prompt = claude_comparer_user_prompt_generator(summary1, summary2)
    
    response = claude.messages.create(
         model=MODEL_CLAUDE,
        max_tokens=10000,
        system=system_prompt_comparer,
        messages=user_prompt)

    summary = response.content[0].text

    display(Markdown(summary))

In [ ]:
await compare_resumes_llama("https://www.reneesingh.com", "https://edwarddonner.com")

In [ ]:
await compare_resumes_claude("https://www.reneesingh.com", "https://edwarddonner.com")